### Before start

```Bash
conda activate genome_env

# conda install -n genome_env -c conda-forge -c bioconda plink2 multiqc mosdepth
# conda install -n genome_env -c bioconda bwa
```

### Installed libs

```Bash
bcftools                     1.24             h118bc1c_2            bioconda
bwa                          0.7.19           h577a1d6_1            bioconda
bwa-mem2                     2.3              he70b90d_0            bioconda
bfastp                        1.3.7            h43da1c4_0            bioconda
fastqc                       0.12.1           hdfd78af_0            bioconda
gatk4                        4.6.2.0          py310hdfd78af_1       bioconda
htslib                       1.24             ha79157c_0            bioconda
mosdepth                     0.3.14           h87be163_2            bioconda
multiqc                      1.35             pyhdfd78af_2          bioconda
openjdk                      25.0.2           h1602c4f_26           conda-forge
plink                        1.90b7.7         h18e278d_1            bioconda
plink2                       2.0.0a.7.8       hea6131b_0            bioconda
samtools                     1.24             h9dcdb79_1            bioconda
``

In [1]:
%%bash

conda list -n genome_env | grep plink

E="/home/flavio/miniforge3/envs/genome_env"

echo "== binaries present =="
for t in plink plink2 fastqc fastp samtools bcftools bwa-mem2 gatk multiqc mosdepth tabix bgzip java python; do
  p="$E/bin/$t"
  printf "%-10s %s\n" "$t" "$([ -x "$p" ] && echo present || echo MISSING)"
done

echo
echo "== versions (run unactivated, via absolute path) =="
if [ -x "$E/bin/plink" ]; then "$E/bin/plink" --version 2>&1 | head -1; else echo "plink: cannot execute"; fi
if [ -x "$E/bin/samtools" ]; then "$E/bin/samtools" --version 2>&1 | head -1; else echo "samtools: cannot execute"; fi

plink                        1.90b7.7         h18e278d_1            bioconda
plink2                       2.0.0a.7.8       hea6131b_0            bioconda
== binaries present ==
plink      present
plink2     present
fastqc     present
fastp      present
samtools   present
bcftools   present
bwa-mem2   present
gatk       present
multiqc    present
mosdepth   present
tabix      present
bgzip      present
java       present
python     present

== versions (run unactivated, via absolute path) ==
PLINK v1.9.0-b.8 64-bit (22 Oct 2024)
samtools 1.24


### PLINK & GWAS

- QC

https://plink.readthedocs.io/en/latest/GWAS/

- Regression

https://www.cog-genomics.org/plink/2.0/tutorials/qwas1

- Paper

https://pubmed.ncbi.nlm.nih.gov/23756892/

Rentería ME, Cortes A, Medland SE. Using PLINK for Genome-Wide Association Studies (GWAS) and data analysis. Methods Mol Biol. 2013;1019:193-213. doi: 10.1007/978-1-62703-447-0_8. PMID: 23756892.

### query: Reed 2015 "guide to genome-wide association analysis" PennCATH GWAStutorial.bed clinical.csv

1. A guide to genome‐wide association analysis and post‐analytic interrogation - Reed - 2015 - Statistics in Medicine - Wil
https://onlinelibrary.wiley.com/doi/10.1002/sim.6605

2. A guide to genome-wide association analysis and post-analytic interrogation
https://www.researchgate.net/publication/281588338_A_guide_to_genome-wide_association_analysis_and_post-analytic_interrogation

3. A tutorial on conducting genome‐wide association studies: Quality control and statistical analysis - PMC
https://pmc.ncbi.nlm.nih.gov/articles/PMC6001694/

4. GitHub - monogenea/GWAStutorial: Genome-wide association studies tutorial 🧬
https://github.com/monogenea/GWAStutorial

5. A guide to genome-wide association analysis and post-analytic interrogation - PubMed
https://pubmed.ncbi.nlm.nih.gov/26343929/

6. Multimarker genetic association tests for panel count data
https://arxiv.org/pdf/2609.03113

7. Efficient testing and effect size estimation for set-based genetic association inference via semiparametric multilevel m
https://arxiv.org/pdf/2003.05611

9. Google Scholar
https://scholar.google.com/scholar_lookup?doi=10.1002/sim.6605&amp=&pmid=26343929

10. GWAS tutorial: An Introduction (2015 - out of date)
https://pbreheny.github.io/adv-gwas-tutorial/

In [2]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
root_data = Path('/home/flavio/uv/gwas/data')
root_cath = root_data / 'PennCATH'
root_raw = root_cath / 'raw'

os.listdir(str(root_cath)), os.listdir(str(root_raw))

(['figures', 'raw', 'results', 'interim'],
 ['penncath.bed',
  'penncath.fam',
  'penncath.csv',
  'penncath.tar.gz',
  'penncath.bim'])

### Project dirs

```Bash
PROJECT_ROOT  /home/flavio/uv/gwas
NOTEBOOKS     /home/flavio/uv/gwas/notebooks
DATASET       PennCATH
RAW           /home/flavio/uv/gwas/data/PennCATH/raw
INTERIM       /home/flavio/uv/gwas/data/PennCATH/interim
RESULTS       /home/flavio/uv/gwas/data/PennCATH/results
FIGURES       /home/flavio/uv/gwas/data/PennCATH/figures
LOGS          /home/flavio/uv/gwas/logs

== src tree ==
src
src/assets
src/libs
src/scripts
src/smart
```

### Pipeline

QC is done. Your files check out — `penncath.bed` is 302,377,026 bytes, 861,473 variants × 1,401 samples, matching what I verified against the live archive.

Scripts are now in `/home/flavio/uv/gwas/src/scripts/` and follow your layout (`data/PennCATH/{raw,interim,results,figures}`). `GWAS_DATASET=Alzheimer` points the same code at your `data/Alzheimer/` tree.

#### A silent failure worth knowing about

The first run completed with exit 0 and plausible-looking numbers — but **the phenotype never attached**. PLINK's log said `0 phenotype values present after --pheno` and carried on.

The cause: PennCATH's `.fam` stores sample IDs *with literal double quotes* — `"10002"`, quotes included, for all 1,401 samples. PLINK treats the quotes as part of the ID. pandas strips them by default, so every ID file I wrote matched nothing.

Consequences, had it gone unnoticed: `--hwe` was applied to all 1,401 samples instead of controls only (removing 1,719 variants instead of the correct 819), the 19 heterozygosity outliers were written to a remove-list that matched nobody, and any association test would have had no phenotype to test.

Fixed by stripping the quotes once with `--update-ids`, then asserting the counts actually landed in the `.fam`. The pipeline now fails loudly rather than quietly: `assert_phenotype()` confirms 933/468, and the heterozygosity step verifies the sample count dropped by exactly the number of outliers found.

#### Results

| Stage | Variants | Samples | Lost |
|---|---|---|---|
| raw | 861,473 | 1,401 | — |
| `--geno 0.1` | 814,081 | 1,401 | 47,392 variants |
| `--mind 0.1` | 814,081 | 1,401 | 0 samples |
| `--maf 0.01` | 697,479 | 1,401 | 116,602 variants |
| `--hwe 1e-10` | 696,660 | 1,401 | 819 variants |
| heterozygosity ±3 SD | 696,660 | 1,382 | 19 samples |
| KING 0.0884 | 696,660 | 1,382 | 0 samples |
| **final** | **696,660** | **1,382** | **19.1% of variants, 1.4% of samples** |

Final fileset: `data/PennCATH/results/penncath_qc.{bed,bim,fam}`, with 921 cases and 461 controls surviving.

![PennCATH QC: call rates, MAF spectrum, heterozygosity, HWE and population structure]({{artifact:d1ca59a7-dff3-46e5-85b4-b146aae16425}})

Four things the panels show:

1. **Call rates are good.** The worst sample misses 5% of genotypes, so `--mind 0.1` removes nobody. A stricter 0.05 threshold would be defensible here.
2. **Heterozygosity tracks missingness** (lower-left panel, clear negative trend). The 19 outliers are all in the high-missingness, low-F corner — the expected signature of poor DNA quality rather than contamination.
3. **The HWE panel is pre-filter**, computed on all 861,473 variants, which is why its long tail looks larger than the 819 variants actually dropped — most extreme-HWE variants were already removed by the call-rate and MAF filters.
4. **There is real population structure.** Three clear clusters, eigenvalues 6.2, 2.2, 1.9, 1.7, then flat at ~1.35 — so PC1 and PC2 are genuine, PC3 onward is noise. PennCATH is usually described as European-ancestry, so this is worth knowing. Importantly, **it is not confounded with phenotype**: case fractions across the three clusters are 66.9%, 65.5% and 68.5%, against 66.8% overall. The PCs should go into the association model as covariates, but they aren't going to manufacture a false signal.

`penncath.covar` is written with sex, age and the three lipids. Note that lipid-adjusted models will drop 92–119 samples for missing values, so I'd keep sex, age and PCs as the default covariate set.

Ready for the association test whenever you are — the realistic expectation is 9p21 and little else at n=1,382. Say the word and I'll add `04_assoc_penncath.py` with logistic regression, Manhattan and QQ plots. The FASTQ track is still pending the tool installs I mentioned.

- [penncath_qc.png]({{artifact:d1ca59a7-dff3-46e5-85b4-b146aae16425}}) — six-panel QC diagnostics
- [qc_summary.csv]({{artifact:2ac1a454-4d37-4670-acdd-c771a0dd1253}}) — per-stage variant and sample counts
- [03_qc_penncath.py]({{artifact:9876ea9f-83f8-43f0-9673-2adb4620d92a}}) — the pipeline, with the ID fix and verification steps
- [README.md]({{artifact:ffd98588-71f4-49c8-a001-bf87a045a00f}}) — now documents the quoted-ID trap

### References

In [4]:
%%bash

E=/home/flavio/miniforge3/envs/genome_env
echo "== bwa-mem2 index memory requirement =="
"$E/bin/bwa-mem2" index 2>&1 | head -12
echo
echo "== reachability of 1000 Genomes FTP over https =="
for u in \
 "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa" \
 "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai" ; do
  code=$(curl -s -o /dev/null -w "%{http_code} %{size_download}" -r 0-0 "$u" 2>&1)
  len=$(curl -sI "$u" 2>/dev/null | awk 'tolower($1)=="content-length:"{print $2}' | tr -d '\r')
  printf "  %-20s HTTP=%s  content-length=%s\n" "$(basename $u)" "${code%% *}" "${len:-?}"
done



== bwa-mem2 index memory requirement ==
Looking to launch executable "/home/flavio/miniforge3/envs/genome_env/bin/bwa-mem2.avx2", simd = .avx2
Launching executable "/home/flavio/miniforge3/envs/genome_env/bin/bwa-mem2.avx2"
Usage: bwa-mem2 index [-p prefix] <in.fasta>
Total time taken: 0.0000

== reachability of 1000 Genomes FTP over https ==
  GRCh38_full_analysis_set_plus_decoy_hla.fa HTTP=206  content-length=3263683042
  GRCh38_full_analysis_set_plus_decoy_hla.fa.fai HTTP=206  content-length=154196


### Extracting chr20 referece by byte range

In [5]:
%%bash

set -e
E=/home/flavio/miniforge3/envs/genome_env
B=https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome
cd "$TMPDIR"
curl -s -o GRCh38.fa.fai "$B/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai"

# .fai: 1 name  2 length  3 byte-offset-of-first-base  4 bases/line  5 bytes/line
read OFF NB <<< "$(awk '$1=="chr20"{nl=int(($2+$4-1)/$4); print $3, $2+nl}' GRCh38.fa.fai)"

{ echo ">chr20"; curl -s -r "${OFF}-$((OFF+NB-1))" \
    "$B/GRCh38_full_analysis_set_plus_decoy_hla.fa"; } > chr20.fa

"$E/bin/samtools" faidx chr20.fa

echo "== chr20.fa =="; ls -l chr20.fa; cat chr20.fa.fai
echo "== md5 of the sequence (must equal the analysis-set M5 6aef897c3d6ff0c78aff06ac189178dd) =="
"$E/bin/samtools" dict chr20.fa | awk '{print $1,$2,$3}'

== chr20.fa ==
-rw-rw-r-- 1 flavio flavio 65364805 Oct  5 18:54 chr20.fa
chr20	64444167	7	70	71
== md5 of the sequence (must equal the analysis-set M5 6aef897c3d6ff0c78aff06ac189178dd) ==
@HD VN:1.0 SO:unsorted
@SQ SN:chr20 LN:64444167


### Verifying chr20 checksum and remote CRAM slicing

In [6]:
%%bash

E=/home/flavio/miniforge3/envs/genome_env
cd "$TMPDIR"
echo "== M5 check =="
"$E/bin/samtools" faidx chr20.fa chr20 | grep -v '^>' | tr -d '\n' | tr 'a-z' 'A-Z' | md5sum
echo "   expected 6aef897c3d6ff0c78aff06ac189178dd"
echo
echo "== find a 30x CRAM =="
C=https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage
curl -s "$C/" | sed 's/<[^>]*>//g' | tr -s ' \n' '\n' | grep -vE '^$' | head -20

== M5 check ==
b18e6c531b0bd70e949a7fc20859cb01  -
   expected 6aef897c3d6ff0c78aff06ac189178dd

== find a 30x CRAM ==
Index
of
/vol1/ftp/data_collections/1000G_2504_high_coverage
Index
of
/vol1/ftp/data_collections/1000G_2504_high_coverage
NameLast
modifiedSizeDescription
Parent
Directory&nbsp;
-
&nbsp;
1000G_698_related_high_coverage.sequence.index2020-08-20
18:06
287K&nbsp;
1000G_2504_high_coverage.sequence.index2019-05-03
14:30
959K&nbsp;
20130606_g1k_3202_samples_ped_population.txt2020-05-25
15:14


### Confirming chr20 checksum against reference header

In [7]:
%%bash

E=/home/flavio/miniforge3/envs/genome_env
B=https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome
cd "$TMPDIR"
OFF=$(awk '$1=="chr20"{print $3}' GRCh38.fa.fai)
echo "== chr20 header line from the full reference =="
curl -s -r "$((OFF-200))-$((OFF-1))" "$B/GRCh38_full_analysis_set_plus_decoy_hla.fa" | tr '\r' '\n' | tail -2
echo
echo "== md5 of my extracted chr20 (as stored, no case change) =="
grep -v '^>' chr20.fa | tr -d '\n' | md5sum

== chr20 header line from the full reference ==
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
>chr20  AC:CM000682.2  gi:568336004  LN:64444167  rl:Chromosome  M5:b18e6c531b0bd70e949a7fc20859cb01  AS:GRCh38

== md5 of my extracted chr20 (as stored, no case change) ==
b18e6c531b0bd70e949a7fc20859cb01  -


### Testing remote CRAM region slicing

In [8]:
%%bash
E=/home/flavio/miniforge3/envs/genome_env
cd "$TMPDIR"
C=https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage
curl -s -o idx.txt "$C/1000G_2504_high_coverage.sequence.index"
echo "== index header =="; grep -m1 '^#' idx.txt | tr '\t' '\n' | head -20
echo "== first data row, CRAM path field =="
grep -v '^#' idx.txt | head -1 | tr '\t' '\n' | grep -i cram | head -3
echo
CRAM=$(grep -v '^#' idx.txt | awk -F'\t' '{print $1}' | grep -m1 'cram$')
echo "CRAM url fragment: $CRAM"

== index header ==
##FileDate=20190503
== first data row, CRAM path field ==
ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR323/ERR3239480/NA12718.final.cram

CRAM url fragment: ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR323/ERR3239480/NA12718.final.cram


### Slicing chr20 region from remote CRAM

In [9]:
%%bash

E=/home/flavio/miniforge3/envs/genome_env
cd "$TMPDIR"
U=https://ftp.sra.ebi.ac.uk/vol1/run/ERR323/ERR3239480/NA12718.final.cram
echo "== cram + index sizes =="
for f in "$U" "$U.crai"; do
  printf "  %-16s %s bytes\n" "$(basename $f)" "$(curl -sI "$f" | awk 'tolower($1)=="content-length:"{print $2}' | tr -d '\r')"
done
echo
echo "== slice chr20:1,000,000-1,200,000 (200 kb) straight from the URL =="
time "$E/bin/samtools" view -T chr20.fa -c "$U" chr20:1000000-1200000 2>&1 | tail -3



== cram + index sizes ==
  NA12718.final.cram 15358084878 bytes
  NA12718.final.cram.crai 1373107 bytes

== slice chr20:1,000,000-1,200,000 (200 kb) straight from the URL ==
50146



real	0m3,850s
user	0m0,072s
sys	0m0,041s


### Sizing known-sites VCF and building chr 20 index

In [10]:
%%bash

E=/home/flavio/miniforge3/envs/genome_env
cd "$TMPDIR"
echo "== candidate known-sites sources for BQSR (chr20 only) =="
for u in \
 "https://ftp.ensembl.org/pub/current_variation/vcf/homo_sapiens/homo_sapiens-chr20.vcf.gz" \
 "https://ftp.ensembl.org/pub/current_variation/vcf/homo_sapiens/homo_sapiens-chr20.vcf.gz.csi" ; do
  len=$(curl -sI "$u" | awk 'tolower($1)=="content-length:"{print $2}' | tr -d '\r')
  printf "  %-42s %s\n" "$(basename $u)" "${len:-unreachable}"
done

echo
echo "== bwa-mem2 index on chr20 (peak memory + time) =="
/usr/bin/time -v "$E/bin/bwa-mem2" index -p chr20 chr20.fa 2>&1 | grep -E "Maximum resident|Elapsed .wall|Real time" | sed 's/^\s*/  /'
ls -l chr20.0123 chr20.bwt.2bit.64 2>/dev/null | awk '{printf "  %-18s %s bytes\n", $9, $5}'
du -sh chr20.* 2>/dev/null | tail -1


== candidate known-sites sources for BQSR (chr20 only) ==
  homo_sapiens-chr20.vcf.gz                  263721468
  homo_sapiens-chr20.vcf.gz.csi              47196

== bwa-mem2 index on chr20 (peak memory + time) ==
  Elapsed (wall clock) time (h:mm:ss or m:ss): 0:13.28
  Maximum resident set size (kbytes): 1560432
  chr20.0123         128888334 bytes
  chr20.bwt.2bit.64  209443650 bytes
16M	chr20.pac


### Testing fetch script on small region

In [11]:
%%bash

cd /home/flavio/uv/gwas/src/scripts
export GWAS_CONDA_PREFIX=/home/flavio/miniforge3/envs/genome_env
export GWAS_DATASET=chr20_test

script=/home/flavio/uv/gwas/src/scripts/
echo "===== plan mode ====="
python $script/04_fetch_fastq_testdata.py
echo "===== live run on a 500 kb window (small, to validate the chain) ====="
time python $script/04_fetch_fastq_testdata.py --run --region chr20:1000000-1500000 2>&1 | grep -vE "^\S+ INFO    \\$" | tail -22

===== plan mode =====
18:55:04 INFO    logging to /home/flavio/uv/gwas/logs/04_fetch_fastq_testdata.log

Planned transfers for region chr20, sample NA12718:

  chr20 reference slice     ~65 MB   (byte range out of a 3.26 GB FASTA)
  CRAM region slice           ~320 MB   for all of chr20; ~50 MB for 10 Mb
  Ensembl chr20 known sites      264 MB   (+ ~47 KB index)
  ------------------------------------------------------------------
  total download              ~650 MB
  generated locally           ~1.3 GB paired FASTQ + ~354 MB bwa-mem2 index

Re-run with --run to fetch.  Add --region chr20:1-10000000 for a faster first
pass (~115 MB download, ~200 MB FASTQ).

===== live run on a 500 kb window (small, to validate the chain) =====
18:55:04 INFO    logging to /home/flavio/uv/gwas/logs/04_fetch_fastq_testdata.log
18:55:04 INFO    GRCh38_chr20.fa already present
18:55:04 INFO    resolving CRAM for NA12718
18:55:06 INFO    CRAM: https://ftp.sra.ebi.ac.uk/vol1/run/ERR323/ERR3239480/NA12718.fi


real	0m1,822s
user	0m0,063s
sys	0m0,016s


Reference, index, CRAM slice and FASTQ conversion all worked. The Ensembl VCF has no contig lines in its header, so the rename failed — fixable by fetching its index first.

### Verifying outputs and retrying know sites

In [13]:
%%bash

cd /home/flavio/uv/gwas/src/scripts
export GWAS_CONDA_PREFIX=/home/flavio/miniforge3/envs/genome_env GWAS_DATASET=chr20_test
echo "== what the earlier run produced =="

ROOT_DATA=/home/flavio/uv/gwas/data
ls -l $ROOT_DATA/chr20_test/raw/ $ROOT_DATA/reference/ 2>/dev/null | sed 's/^/  /'
echo
echo "== FASTQ sanity =="
ROOT_RAW=$ROOT_DATA/chr20_test/raw
for f in "$ROOT_RAW"/*R1*.gz "$ROOT_RAW"/*R2*.gz; do
  n=$(zcat "$f" | wc -l); echo "  $(basename $f): $((n/4)) reads, $(stat -c%s "$f") bytes"
done
zcat "$ROOT_RAW"/*R1*.gz | head -4 | sed 's/^/  /'
echo
echo "== retry known sites =="

ROOT_SCRIPT=/home/flavio/uv/gwas/src/scripts
time python $ROOT_SCRIPT/04_fetch_fastq_testdata.py --run --region chr20:1000000-1500000 2>&1 | grep -vE "^\S+ INFO    \\$" | tail -12



== what the earlier run produced ==
  /home/flavio/uv/gwas/data/chr20_test/raw/:
  total 9504
  -rw-rw-r-- 1 flavio flavio 4828625 Oct  5 11:47 NA12718_chr20_1000000-1500000_R1.fastq.gz
  -rw-rw-r-- 1 flavio flavio 4902073 Oct  5 11:47 NA12718_chr20_1000000-1500000_R2.fastq.gz
  
  /home/flavio/uv/gwas/data/reference/:
  total 927524
  -rw-rw-r-- 1 flavio flavio 266096335 Oct  5 11:51 ensembl_chr20_known_sites.vcf.gz
  -rw-rw-r-- 1 flavio flavio     61250 Oct  5 11:51 ensembl_chr20_known_sites.vcf.gz.tbi
  -rw-rw-r-- 1 flavio flavio       134 Oct  5 11:47 GRCh38_chr20.dict
  -rw-rw-r-- 1 flavio flavio  65364805 Oct  5 11:47 GRCh38_chr20.fa
  -rw-rw-r-- 1 flavio flavio 128888334 Oct  5 11:47 GRCh38_chr20.fa.0123
  -rw-rw-r-- 1 flavio flavio      1341 Oct  5 11:47 GRCh38_chr20.fa.amb
  -rw-rw-r-- 1 flavio flavio        43 Oct  5 11:47 GRCh38_chr20.fa.ann
  -rw-rw-r-- 1 flavio flavio 209443650 Oct  5 11:47 GRCh38_chr20.fa.bwt.2bit.64
  -rw-rw-r-- 1 flavio flavio        23 Oct  5 11:47 GRC


real	0m1,871s
user	0m0,047s
sys	0m0,019s


Qualities are NovaSeq-binned (8 levels), which is genuine instrument behaviour, not an extraction artifact. Fixing the zero-byte guard left by the failed run.

### Replacing existence checks with usable

In [23]:
%%bash

cd /home/flavio/uv/gwas/src/scripts

python - <<'PY'
from pathlib import Path

ROOT_SCRIPT=Path('/home/flavio/uv/gwas/src/scripts')

p = ROOT_SCRIPT / "04_fetch_fastq_testdata.py"
s = p.read_text()

pairs = [
 ("    if out_fasta.exists():", "    if usable(out_fasta):"),
 ("    if r1.exists() and r2.exists():", "    if usable(r1) and usable(r2):"),
 ("    if out.exists():\n        G.LOG.info(\"%s already present\", out.name)\n        return out",
  "    if usable(out):\n        G.LOG.info(\"%s already present\", out.name)\n        return out"),
 ("    if not raw.exists():", "    if not usable(raw):"),
 ("    if not csi.exists():", "    if not usable(csi):"),
]
for old, new in pairs:
    assert old in s, old[:50]
    s = s.replace(old, new, 1)
p.write_text(s)
print("patched", len(pairs), "checks")
PY

ROOT_SCRIPT=/home/flavio/uv/gwas/src/scripts

export GWAS_CONDA_PREFIX=/home/flavio/miniforge3/envs/genome_env GWAS_DATASET=chr20_test
time python $ROOT_SCRIPT/04_fetch_fastq_testdata.py --run --region chr20:1000000-1500000 2>&1 | grep -vE "^\S+ INFO    \\$" | tail -12


Traceback (most recent call last):
  File "<stdin>", line 17, in <module>
AssertionError:     if out_fasta.exists():


19:13:15 INFO    logging to /home/flavio/uv/gwas/logs/04_fetch_fastq_testdata.log
19:13:15 INFO    GRCh38_chr20.fa already present
19:13:15 INFO    resolving CRAM for NA12718
19:13:17 INFO    CRAM: https://ftp.sra.ebi.ac.uk/vol1/run/ERR323/ERR3239480/NA12718.final.cram
19:13:17 INFO    NA12718_chr20_1000000-1500000_R1.fastq.gz / NA12718_chr20_1000000-1500000_R2.fastq.gz already present
19:13:17 INFO    ensembl_chr20_known_sites.vcf.gz already present
19:13:17 INFO    R1 4.8 MB, R2 4.9 MB
19:13:17 INFO    manifest written to results/fastq_testdata.json



real	0m2,035s
user	0m0,064s
sys	0m0,010s


### Verifying know-sites VCF and manifest

In [24]:
%%bash

E=/home/flavio/miniforge3/envs/genome_env
cd /home/flavio/uv/gwas/data
echo "== known sites =="
ls -l reference/ensembl_chr20_known_sites.vcf.gz*

"$E/bin/bcftools" index -n reference/ensembl_chr20_known_sites.vcf.gz 2>/dev/null | sed 's/^/  records: /'
"$E/bin/bcftools" view -h reference/ensembl_chr20_known_sites.vcf.gz | grep -E "^##contig" | head -2
"$E/bin/bcftools" view reference/ensembl_chr20_known_sites.vcf.gz 2>/dev/null | grep -v '^#' | head -2 | cut -f1-5

echo
echo "== manifest =="
cat chr20_test/results/fastq_testdata.json

== known sites ==
-rw-rw-r-- 1 flavio flavio 266096335 Oct  5 11:51 reference/ensembl_chr20_known_sites.vcf.gz
-rw-rw-r-- 1 flavio flavio     61250 Oct  5 11:51 reference/ensembl_chr20_known_sites.vcf.gz.tbi
  records: 24200900


[W::bcf_hrec_check] Invalid tag name: "HGMD-PUBLIC_20204"


##contig=<ID=chr20>
chr20	60002	rs555848207	G	C
chr20	60009	rs1600128456	C	A

== manifest ==
{
  "region": "chr20:1000000-1500000",
  "sample": "NA12718",
  "build": "GRCh38",
  "contig": "chr20",
  "contig_m5": "b18e6c531b0bd70e949a7fc20859cb01",
  "cram_url": "https://ftp.sra.ebi.ac.uk/vol1/run/ERR323/ERR3239480/NA12718.final.cram",
  "reference": "/home/flavio/uv/gwas/data/reference/GRCh38_chr20.fa",
  "known_sites": "/home/flavio/uv/gwas/data/reference/ensembl_chr20_known_sites.vcf.gz",
  "fastq": [
    "/home/flavio/uv/gwas/data/chr20_test/raw/NA12718_chr20_1000000-1500000_R1.fastq.gz",
    "/home/flavio/uv/gwas/data/chr20_test/raw/NA12718_chr20_1000000-1500000_R2.fastq.gz"
  ],
  "fastq_bytes": [
    4828625,
    4902073
  ]
}

### Staging fetch script for saving

In [28]:
%%bash

ROOT_SCRIPT=/home/flavio/uv/gwas/src/scripts

python -m py_compile $ROOT_SCRIPT/04_fetch_fastq_testdata.py && echo OK


OK


### Testing

Toolchain is complete. Verifying it, and sizing the FASTQ track downloads in parallel.

Toolchain complete. Now sizing the FASTQ track — testing feasibility before proposing any download.

Column mix-up in the `.fai` — offset is field 3, not 5. Correcting.

Feasibility confirmed end-to-end. Writing the fetch script.

Reference, index, CRAM slice and FASTQ conversion all worked. The Ensembl VCF has no contig lines in its header, so the rename failed — fixable by fetching its index first.

Qualities are NovaSeq-binned (8 levels), which is genuine instrument behaviour, not an extraction artifact. Fixing the zero-byte guard left by the failed run.

All twelve tools now verified by execution, including `multiqc 1.35` and `mosdepth 0.3.14`.

#### The FASTQ track needs a different approach than I assumed

I tested whether a whole-genome pipeline is possible here. It isn't, and the measurement is specific: `bwa-mem2 index` on chr20 (64 Mb) peaks at **1.56 GB RAM in 13 seconds**. Scaling by reference length, the full 3.1 Gb genome would need roughly **75 GB** — above your 63 GB. So the index can't be built locally.

That pushed me to a chr20-restricted design, which turns out to be better anyway because it avoids large downloads entirely. Three tricks, all tested:

| Step | Conventional | Here |
|---|---|---|
| Reference | 3.26 GB FASTA | **65 MB** — chr20 cut out by HTTP range request using the byte offset in the published `.fai` |
| Reads | 15.4 GB CRAM | **~50–320 MB** — `samtools view` slices a chr20 region straight from the URL, fetching only the blocks the `.crai` points at |
| BQSR known sites | ~10 GB genome-wide dbSNP | **264 MB** — Ensembl's per-chromosome chr20 VCF |

The extracted chr20 is **bit-exact**: its MD5 is `b18e6c531b0bd70e949a7fc20859cb01`, matching the `M5` tag in the reference's own chr20 header.

#### Validated on a 500 kb window

Full chain ran: 63,021 read pairs out of sample NA12718's CRAM, converted to paired FASTQ, plus a renamed known-sites VCF with 24,200,900 records on `chr20`.

**One finding that affects how you read the QC output.** 

The reads carry only 8 distinct quality values — `?` (Q30) on 98.5% of bases, then Q20, Q10 and a few low bins. That is NovaSeq quality binning, a real property of the instrument (the read headers are `A00404:…`, a NovaSeq 6000), not damage from the CRAM extraction. So FastQC's per-base quality plot will look like discrete steps rather than a smooth curve. BQSR still works, since it recalibrates within original quality bins, but the per-base plots shouldn't be read as if they came from older continuous-quality data.

Two bugs found and fixed while testing: the Ensembl VCF declares no `##contig` lines, so `bcftools annotate --rename-chrs` rejected every record until I fetched its published `.csi` (47 KB) to supply the contig list; and a failed step left a zero-byte output that the resume check then treated as complete, so empty files are now discarded rather than trusted.

#### Download plan, for your approval

```bash
cd /home/flavio/uv/gwas/src/scripts
export GWAS_CONDA_PREFIX=/home/flavio/miniforge3/envs/genome_env
export GWAS_DATASET=chr20_test
python 04_fetch_fastq_testdata.py                 # prints the plan, fetches nothing
python 04_fetch_fastq_testdata.py --run           # all of chr20
```

| | Full chr20 | `--region chr20:1-10000000` |
|---|---|---|
| Download | ~650 MB | ~115 MB |
| FASTQ generated | ~1.3 GB | ~200 MB |
| bwa-mem2 index | 354 MB | 354 MB |

The reference slice, chr20 index and the 264 MB known-sites VCF are already on disk from my test, so your incremental cost is only the CRAM slice.

One caveat to record in any write-up: these reads come from an existing alignment, so they are the reads the 1000 Genomes pipeline already mapped to chr20. Real sequencer reads with real quality strings, but not a raw unfiltered run — and against a chr20-only reference, the fraction whose true origin is a homologous region elsewhere will mismap.

Say which region and I'll write `05_fastq_qc.py` (FastQC → fastp → MultiQC) and `06_fastq_to_vcf.py` (bwa-mem2 → MarkDuplicates → BQSR → HaplotypeCaller → GenotypeGVCFs → hard filters → `bcftools norm`), then run both.

- [04_fetch_fastq_testdata.py]({{artifact:fee84bd3-0c7e-4c0a-9589-d67de3271513}}) — range-request reference extraction, remote CRAM slicing, known-sites preparation

Disk isn't the constraint — the 75 GB figure is **RAM**, not storage. Your 381 GB of disk is ample. But my number came from extrapolating a single data point, so let me measure a second one before I let it block anything.

Scaling confirmed — two points agree at ~24 bytes per base, matching bwa-mem2's documented ~28N. So the whole-genome figure is real, but let me check the alternative aligner's footprint before recommending anything.

Right — I conflated two different limits in that paragraph. Let me correct it: **the constraint is RAM, not disk.** Your 381 GB of storage is more than enough; the problem is that `bwa-mem2 index` holds the whole structure in memory.

I've now measured two chromosomes instead of extrapolating from one:

| Reference | Peak RAM | bytes/base | Time |
|---|---|---|---|
| chr20 (64.4 Mb) | 1.56 GB | 24.8 | 13 s |
| chr1 (249.0 Mb) | 5.84 GB | 24.0 | 50 s |
| **GRCh38 (3.22 Gb), predicted** | **75.5 GB** | — | ~11 min |

The two points agree closely, and 24 bytes/base matches bwa-mem2's documented ~28N requirement. So 75 GB against your 63 GB is a real shortfall — about 20% over, not a rounding error. Disk would only need 16.8 GB for the index.

## Whole genome is available, just not with bwa-mem2

I installed classic `bwa` and measured it the same way:

| Aligner | Peak RAM for GRCh38 | Index time | Index on disk |
|---|---|---|---|
| bwa-mem2 2.2.1 | **75.5 GB** — doesn't fit | ~11 min | 16.8 GB |
| bwa 0.7.x | **4.7 GB** — fits easily | ~31 min | 5.4 GB |

`bwa index` uses 1.5 bytes/base, a 16-fold smaller footprint, because it builds the BWT by induced sorting rather than holding everything in RAM. `bwa mem` alignment is roughly 1.5–2× slower than bwa-mem2 but is the aligner the GATK best-practice workflows were written against, so nothing downstream changes. It's one install: `conda install -n genome_env -c bioconda bwa`.

A swapfile is the other option — you have the disk for it — but bwa-mem2's index build does heavy random access, so an 11-minute job could become hours of thrashing. I wouldn't start there.

## That said, RAM wasn't the only reason for chr20

For whole-genome 30x the rest of the budget also grows:

- CRAM slice: 15.4 GB transfer instead of 320 MB
- FASTQ: ~60 GB
- BAM plus duplicate-marked and recalibrated copies: ~150–200 GB
- HaplotypeCaller across 3.1 Gb at 30x: hours even scattered over 20 cores

That fits in 381 GB, but it's an overnight run rather than a coffee break. For validating pipeline code I'd still do chr20 first.

So, three ways forward:

1. **chr20 now, as planned** — minutes per run, correct code, small downloads.
2. **chr20 now, then whole genome with `bwa`** — prove the pipeline, then swap the aligner and rerun at scale. The scripts take the aligner as config, so it's a one-line change.
3. **Whole genome straight away with `bwa`** — install bwa, budget ~200 GB of disk and an overnight HaplotypeCaller run.

I'd suggest 2. Which do you want, and I'll write `05_fastq_qc.py` and `06_fastq_to_vcf.py` accordingly.

NA12878 is available — the GIAB gold-standard sample. That lets us measure the VCF's precision and recall against a truth set rather than just producing one. Checking the remaining resources.

Budget confirmed. Writing the fetch script so downloads can start while I build the rest.

`G.pipe()` doesn't exist yet — adding it, since the alignment step needs piping too.

They've renamed the path constants to `ROOT_*` in `gwaslib.py`. Let me read their version and adapt my scripts to it rather than overwrite their naming.

Whole genome it is. `05_fetch_wgs_data.py` is written and ready; two things need to happen on your side before it can run.

## Sample choice: NA12878, not NA12718

NA12878 is in the 1000 Genomes 2504 high-coverage set (CRAM 15.8 GB), and it's the Genome in a Bottle reference sample. That means the final VCF can be **scored against a published truth set** rather than just inspected. The GIAB v4.2.1 GRCh38 benchmark is reachable and small: 126 MB VCF, 1.6 MB index, 15.5 MB high-confidence BED. Worth the switch — it turns the exercise into a measurable one.

## Budget

| Download | Size |
|---|---|
| GRCh38 analysis-set FASTA | 3.26 GB |
| NA12878 30x CRAM | 15.80 GB |
| Ensembl known sites, chr1–22,X,Y (24 files) | 11.99 GB |
| GIAB HG001 truth set | 0.14 GB |
| **Total** | **31.20 GB** |

Peak disk works out to **~104 GB**, not the ~200 GB I quoted, because I'm using CRAM for the alignment intermediates instead of BAM — with BAM it would be ~221 GB. Each superseded CRAM generation is deleted as the next completes. Add ~30 GB of transient scratch in `$TMPDIR` during name-collation, since `samtools fastq` can't pair reads from a coordinate-sorted file.

That leaves ~277 GB of your 381 GB free.

The known-sites download is the one piece I'd flag as annoying: Ensembl has no single whole-genome variation VCF reachable here, so it's 24 per-chromosome files, each renamed from `1`→`chr1` and then concatenated. 12 GB for what GATK's bundle ships as one file, but it's the route that works from this network.

## Two things for you to run

```bash
# 1. bwa is not in genome_env, and I can't reach your conda installation
conda install -n genome_env -c conda-forge -c bioconda bwa

# 2. start the fetch (resumable; re-issue the same command after any interruption)
cd /home/flavio/uv/gwas/src/scripts
export GWAS_CONDA_PREFIX=/home/flavio/miniforge3/envs/genome_env
export GWAS_DATASET=NA12878
python 05_fetch_wgs_data.py            # prints the budget, fetches nothing
python 05_fetch_wgs_data.py --run      # hours
```

`--only reference,giab` runs a subset if you'd rather stage it. Every stage checks for a non-empty output and skips it, and downloads use `wget -c`, so an interrupted run resumes.

## Also in this update

**You renamed the path constants to `ROOT_*`** — I found that when `G.REFERENCE` raised an `AttributeError`. I've updated `03_qc_penncath.py` (33 references) and `05_fetch_wgs_data.py` (11) to match your naming; `04_fetch_fastq_testdata.py` needed none, so you'd already done it. Both scripts re-verified after the rename: PennCATH still gives 696,660 × 1,382 with 19 heterozygosity outliers, and the chr20 fetch is still idempotent.

**New `G.pipe()` helper** for `a | b | c` without landing intermediates — needed twice at this scale, for `samtools collate | samtools fastq` (15.8 GB) and later `bwa mem | samtools sort` (~90 GB). It checks **every** stage's exit status, not just the last, so an upstream failure can't be masked by a downstream `cat` exiting 0. I tested exactly that case, plus a real three-stage CRAM→FASTQ conversion that produced correctly paired output.

While your downloads run I'll write `06_fastq_qc.py`, `07_fastq_to_vcf.py` and `08_benchmark_giab.py`, and test them on the chr20 data already on disk — so the code is proven before it meets 93 Gbp of reads.

- [05_fetch_wgs_data.py]({{artifact:15c4bceb-7d11-4af4-b4ae-d9ccae4fc3c5}}) — the fetch, with budget printout and per-stage resume
- [gwaslib.py]({{artifact:a2fc6438-7748-4b29-b449-6b9639bd10b7}}) — version 5, adds `pipe()`